# AdventureWorks: Sales and Purchasing Star Schemas
**Author:** Bala Sai Kiran Reddy Telluri

## Objective
Build a purchasing star and connect it to the sales star through a shared product dimension. The notebook follows a Bronze–Silver–Gold workflow and answers two business questions:
- Which five products have the lowest sales-to-purchase margin per unit?
- For which products and months do sales quantities exceed purchase-order quantities?

## Data model
- Sales: gold.fact_sales linked to silver.dim_product and silver.dim_sales_territory.
- Purchasing: gold.fact_purchases linked to silver.dim_product and silver.dim_vendor.
- Both stars share silver.dim_product.

## Sources
Microsoft AdventureWorks CSV files provide the records. The CREATE TABLE definitions in instawdb.sql provide the column names, order and source data types. Raw files are stored in the Unity Catalog volume; Bronze tables remain unchanged during later transformations.

## 1. Inspect the source files
List the files staged in /Volumes/adventureworks/default/raw_data and confirm the
required CSVs and the instawdb.sql schema reference are present before loading.
Raw files stay in the volume; they are not copied into this repository.

In [0]:
display(dbutils.fs.ls("/Volumes/adventureworks/default/raw_data"))

## 2. Bronze: Purchase-order detail
Read the headerless, tab-separated CSV with raw string columns and check the file
structure before saving.
- Verified input: 8,845 rows and 11 columns.
- Target: adventureworks.bronze.stg_purchase_order_detail.

In [0]:
df = spark.read.option("sep", "\t").option("header", "false").csv("/Volumes/adventureworks/default/raw_data/PurchaseOrderDetail.csv")
print("Rows:", df.count())
print("Columns:", len(df.columns))
print(df.columns)
display(df.limit(5))

### Save the raw purchase-order detail
Create the Bronze staging table without overwriting an existing table. The default
save behavior stops with an error if the table already exists, so existing data can
never be replaced by accident. The raw _c0-_c10 column names stay for the Silver
mapping step.

### Verify the saved detail table
Read the table back from the catalog and inspect its dimensions. Saved result:
8,845 rows and 11 columns, matching the input file.

In [0]:
df.write.saveAsTable("adventureworks.bronze.stg_purchase_order_detail")

In [0]:
saved = spark.table("adventureworks.bronze.stg_purchase_order_detail")
print("Saved rows:", saved.count())
print("Saved columns:", len(saved.columns))

### Load purchase-order headers
Read the raw header file and check its structure before saving.
- Verified input: 4,012 rows and 13 columns.
- Target: adventureworks.bronze.stg_purchase_order_header.


In [0]:
df_header = spark.read.option("sep", "\t").option("header", "false").csv("/Volumes/adventureworks/default/raw_data/PurchaseOrderHeader.csv")
print("Rows:", df_header.count())
print("Columns:", len(df_header.columns))


In [0]:
print("Header rows:", df_header.count())
print("Header columns:", len(df_header.columns))

### Save and verify purchase-order headers
Create the staging table only if it is missing, then read it back. Verified saved
result: 4,012 rows and 13 columns.


In [0]:

if not spark.catalog.tableExists("adventureworks.bronze.stg_purchase_order_header"):
    df_header.write.saveAsTable("adventureworks.bronze.stg_purchase_order_header")

header_saved = spark.table("adventureworks.bronze.stg_purchase_order_header")
print("Saved header rows:", header_saved.count())
print("Saved header columns:", len(header_saved.columns))

### Load vendor records
Read the vendor CSV without headers and verify its structure.
- Verified input: 104 rows and 8 columns.
- Target: adventureworks.bronze.dim_vendor.

In [0]:
df_vendor = spark.read.option("sep", "\t").option("header", "false").csv("/Volumes/adventureworks/default/raw_data/Vendor.csv")
print("Vendor rows:", df_vendor.count())
print("Vendor columns:", len(df_vendor.columns))

### Save and verify raw vendor records
Create the staging table only if it is missing, then read it back. Verified saved
result: 104 rows and 8 columns. Official names and types are applied in Silver.

In [0]:

if not spark.catalog.tableExists("adventureworks.bronze.dim_vendor"):
    df_vendor.write.saveAsTable("adventureworks.bronze.dim_vendor")

vendor_saved = spark.table("adventureworks.bronze.dim_vendor")
print("Saved vendor rows:", vendor_saved.count())
print("Saved vendor columns:", len(vendor_saved.columns))

### Load sales-order headers
Load the additional sales header file: the existing sales-detail table has no
order date, which the monthly analysis needs.
- Verified input: 31,465 rows and 26 columns.
- Target: adventureworks.bronze.stg_sales_order_header.

In [0]:
df_sales_header = spark.read.option("sep", "\t").option("header", "false").csv("/Volumes/adventureworks/default/raw_data/SalesOrderHeader.csv")
print("Sales header rows:", df_sales_header.count())
print("Sales header columns:", len(df_sales_header.columns))

### Save and verify sales-order headers
Create the staging table only if it is missing, then read it back. Verified saved
result: 31,465 rows and 26 columns.

In [0]:
if not spark.catalog.tableExists("adventureworks.bronze.stg_sales_order_header"):
    df_sales_header.write.saveAsTable("adventureworks.bronze.stg_sales_order_header")

sales_header_saved = spark.table("adventureworks.bronze.stg_sales_order_header")
print("Saved sales header rows:", sales_header_saved.count())
print("Saved sales header columns:", len(sales_header_saved.columns))

## 3. Silver: Standardise names and types
Create the Silver schema and map the raw purchase-detail columns to the official
names from instawdb.sql, following the CSV column order. Bronze stays unchanged.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS adventureworks.silver")

purchase_detail = spark.table("adventureworks.bronze.stg_purchase_order_detail")
purchase_detail = purchase_detail.toDF(
    "PurchaseOrderID", "PurchaseOrderDetailID", "DueDate",
    "OrderQty", "ProductID", "UnitPrice", "LineTotal",
    "ReceivedQty", "RejectedQty", "StockedQty", "ModifiedDate"
)
print(purchase_detail.columns)

In [0]:
raw_check = spark.table("adventureworks.bronze.stg_purchase_order_detail")
print(raw_check.columns)
raw_check.selectExpr(
    "cast(_c6 as decimal(19,4)) as LineTotal",
    "cast(_c5 as decimal(19,4)) as UnitPrice"
).limit(3).show()


### Apply purchase-detail data types
Convert identifiers to integers, order quantity to smallint, unit price, line
total and the received/rejected/stocked quantities to decimal, and dates to
timestamps. The schema printout and sample rows confirm the transformation.

In [0]:
purchase_detail = spark.table("adventureworks.bronze.stg_purchase_order_detail").selectExpr(
    "cast(_c0 as int) as PurchaseOrderID",
    "cast(_c1 as int) as PurchaseOrderDetailID",
    "cast(_c2 as timestamp) as DueDate",
    "cast(_c3 as smallint) as OrderQty",
    "cast(_c4 as int) as ProductID",
    "cast(_c5 as decimal(19,4)) as UnitPrice",
    "cast(_c6 as decimal(19,4)) as LineTotal",
    "cast(_c7 as decimal(8,2)) as ReceivedQty",
    "cast(_c8 as decimal(8,2)) as RejectedQty",
    "cast(_c9 as decimal(8,2)) as StockedQty",
    "cast(_c10 as timestamp) as ModifiedDate"
)
purchase_detail.printSchema()
display(purchase_detail.limit(5))


### Clean and save purchase-order detail
Apply the required rule: remove records with NULL ProductID or negative LineTotal.
NULL LineTotal stays, since the assignment does not ask to remove it.
- Before: 8,845 rows. After: 8,845 rows. Dropped: 0.
- Saved table: silver.stg_purchase_order_detail.
The rule is implemented and checked; this dataset simply contains no matching
records.

In [0]:
before_count = purchase_detail.count()
purchase_detail_clean = purchase_detail.where(
    "ProductID IS NOT NULL AND (LineTotal IS NULL OR LineTotal >= CAST(0 AS DECIMAL(19,4)))"
)
after_count = purchase_detail_clean.count()
print("Before:", before_count, "After:", after_count, "Dropped:", before_count - after_count)

if not spark.catalog.tableExists("adventureworks.silver.stg_purchase_order_detail"):
    purchase_detail_clean.write.saveAsTable("adventureworks.silver.stg_purchase_order_detail")

saved_detail = spark.table("adventureworks.silver.stg_purchase_order_detail")
print("Saved Silver rows:", saved_detail.count())
saved_detail.printSchema()


Apply official names and types. Check that PurchaseOrderID, VendorID and OrderDate
are populated and that PurchaseOrderID is unique; a duplicated header key would
multiply detail rows in the fact join.
- Saved table: silver.stg_purchase_order_header. Verified rows: 4,012.

In [0]:
purchase_header = spark.table("adventureworks.bronze.stg_purchase_order_header").selectExpr(
    "cast(_c0 as int) as PurchaseOrderID",
    "cast(_c1 as tinyint) as RevisionNumber",
    "cast(_c2 as tinyint) as Status",
    "cast(_c3 as int) as EmployeeID",
    "cast(_c4 as int) as VendorID",
    "cast(_c5 as int) as ShipMethodID",
    "cast(_c6 as timestamp) as OrderDate",
    "cast(_c7 as timestamp) as ShipDate",
    "cast(_c8 as decimal(19,4)) as SubTotal",
    "cast(_c9 as decimal(19,4)) as TaxAmt",
    "cast(_c10 as decimal(19,4)) as Freight",
    "cast(_c11 as decimal(19,4)) as TotalDue",
    "cast(_c12 as timestamp) as ModifiedDate"
)
assert purchase_header.where("PurchaseOrderID IS NULL OR VendorID IS NULL OR OrderDate IS NULL").count() == 0, "Missing header key or date"
assert purchase_header.select("PurchaseOrderID").distinct().count() == purchase_header.count(), "Duplicate order headers"

if not spark.catalog.tableExists("adventureworks.silver.stg_purchase_order_header"):
    purchase_header.write.saveAsTable("adventureworks.silver.stg_purchase_order_header")

print("Silver header rows:", spark.table("adventureworks.silver.stg_purchase_order_header").count())

### Prepare the vendor dimension
Apply official names and types, including the Boolean flags. Check that the key is
unique and required fields are populated. Per instawdb.sql, the vendor key is
BusinessEntityID, referenced by the purchase header's VendorID.
- Saved table: silver.dim_vendor. Verified rows: 104.

In [0]:
vendor = spark.table("adventureworks.bronze.dim_vendor").selectExpr(
    "cast(_c0 as int) as BusinessEntityID",
    "_c1 as AccountNumber",
    "_c2 as Name",
    "cast(_c3 as tinyint) as CreditRating",
    "cast(_c4 as boolean) as PreferredVendorStatus",
    "cast(_c5 as boolean) as ActiveFlag",
    "_c6 as PurchasingWebServiceURL",
    "cast(_c7 as timestamp) as ModifiedDate"
)
assert vendor.where("BusinessEntityID IS NULL OR CreditRating IS NULL OR PreferredVendorStatus IS NULL OR ActiveFlag IS NULL OR ModifiedDate IS NULL").count() == 0, "Vendor type check failed"
assert vendor.select("BusinessEntityID").distinct().count() == vendor.count(), "Duplicate vendor keys"

if not spark.catalog.tableExists("adventureworks.silver.dim_vendor"):
    vendor.write.saveAsTable("adventureworks.silver.dim_vendor")

print("Silver vendor rows:", spark.table("adventureworks.silver.dim_vendor").count())

%md
### Prepare sales-order headers
Apply names and types to all 26 columns and check that SalesOrderID is unique.
The typed header adds OrderDate and TerritoryID for the sales fact.
- Saved table: silver.stg_sales_order_header. Verified rows: 31,465.

In [0]:
sales_header = spark.table("adventureworks.bronze.stg_sales_order_header").selectExpr(
    "cast(_c0 as int) as SalesOrderID",
    "cast(_c1 as tinyint) as RevisionNumber",
    "cast(_c2 as timestamp) as OrderDate",
    "cast(_c3 as timestamp) as DueDate",
    "cast(_c4 as timestamp) as ShipDate",
    "cast(_c5 as tinyint) as Status",
    "cast(_c6 as boolean) as OnlineOrderFlag",
    "_c7 as SalesOrderNumber",
    "_c8 as PurchaseOrderNumber",
    "_c9 as AccountNumber",
    "cast(_c10 as int) as CustomerID",
    "cast(_c11 as int) as SalesPersonID",
    "cast(_c12 as int) as TerritoryID",
    "cast(_c13 as int) as BillToAddressID",
    "cast(_c14 as int) as ShipToAddressID",
    "cast(_c15 as int) as ShipMethodID",
    "cast(_c16 as int) as CreditCardID",
    "_c17 as CreditCardApprovalCode",
    "cast(_c18 as int) as CurrencyRateID",
    "cast(_c19 as decimal(19,4)) as SubTotal",
    "cast(_c20 as decimal(19,4)) as TaxAmt",
    "cast(_c21 as decimal(19,4)) as Freight",
    "cast(_c22 as decimal(19,4)) as TotalDue",
    "_c23 as Comment",
    "_c24 as rowguid",
    "cast(_c25 as timestamp) as ModifiedDate"
)
assert sales_header.where("SalesOrderID IS NULL OR OrderDate IS NULL OR TerritoryID IS NULL").count() == 0, "Missing sales header key/date"
assert sales_header.select("SalesOrderID").distinct().count() == sales_header.count(), "Duplicate sales order headers"

if not spark.catalog.tableExists("adventureworks.silver.stg_sales_order_header"):
    sales_header.write.saveAsTable("adventureworks.silver.stg_sales_order_header")

print("Silver sales header rows:", spark.table("adventureworks.silver.stg_sales_order_header").count())

### Prepare the shared product dimension
Conform the existing Lecture 1 product table to Silver with the official
instawdb.sql types. Empty optional fields in the source (for example SellEndDate)
legitimately become NULL. ProductID uniqueness is checked. Both stars share this
dimension.
- Saved table: silver.dim_product. Verified rows: 504.

In [0]:
product_raw = spark.table("adventureworks.bronze.product")
print("Source columns:", product_raw.columns)
assert "ProductID" in product_raw.columns and "Name" in product_raw.columns, "Product column names differ; paste the list above"

product_dim = product_raw.selectExpr(
    "cast(ProductID as int) as ProductID",
    "Name", "ProductNumber",
    "cast(MakeFlag as boolean) as MakeFlag",
    "cast(FinishedGoodsFlag as boolean) as FinishedGoodsFlag",
    "Color",
    "cast(SafetyStockLevel as smallint) as SafetyStockLevel",
    "cast(ReorderPoint as smallint) as ReorderPoint",
    "cast(StandardCost as decimal(19,4)) as StandardCost",
    "cast(ListPrice as decimal(19,4)) as ListPrice",
    "Size", "SizeUnitMeasureCode", "WeightUnitMeasureCode",
    "cast(nullif(trim(cast(Weight as string)), '') as decimal(8,2)) as Weight",
    "cast(DaysToManufacture as int) as DaysToManufacture",
    "ProductLine", "Class", "Style",
    "cast(nullif(trim(cast(ProductSubcategoryID as string)), '') as int) as ProductSubcategoryID",
    "cast(nullif(trim(cast(ProductModelID as string)), '') as int) as ProductModelID",
    "cast(SellStartDate as timestamp) as SellStartDate",
    "cast(nullif(trim(cast(SellEndDate as string)), '') as timestamp) as SellEndDate",
    "cast(nullif(trim(cast(DiscontinuedDate as string)), '') as timestamp) as DiscontinuedDate",
    "rowguid",
    "cast(ModifiedDate as timestamp) as ModifiedDate"
)
assert product_dim.where("ProductID IS NULL OR Name IS NULL").count() == 0, "Missing product key/name"
assert product_dim.select("ProductID").distinct().count() == product_dim.count(), "Duplicate product keys"

if not spark.catalog.tableExists("adventureworks.silver.dim_product"):
    product_dim.write.saveAsTable("adventureworks.silver.dim_product")

print("Silver product rows:", spark.table("adventureworks.silver.dim_product").count())

### Prepare the sales-territory dimension
Conform the existing Lecture 1 territory table to Silver and check TerritoryID
uniqueness.
- Saved table: silver.dim_sales_territory. Verified rows: 10.

In [0]:
territory_raw = spark.table("adventureworks.bronze.salesterritory")
print("Source columns:", territory_raw.columns)

territory_dim = territory_raw.selectExpr(
    "cast(TerritoryID as int) as TerritoryID",
    "Name", "CountryRegionCode", "`Group`",
    "cast(SalesYTD as decimal(19,4)) as SalesYTD",
    "cast(SalesLastYear as decimal(19,4)) as SalesLastYear",
    "cast(CostYTD as decimal(19,4)) as CostYTD",
    "cast(CostLastYear as decimal(19,4)) as CostLastYear",
    "rowguid",
    "cast(ModifiedDate as timestamp) as ModifiedDate"
)
assert territory_dim.where("TerritoryID IS NULL OR Name IS NULL OR CountryRegionCode IS NULL OR `Group` IS NULL OR SalesYTD IS NULL OR SalesLastYear IS NULL OR CostYTD IS NULL OR CostLastYear IS NULL OR ModifiedDate IS NULL").count() == 0, "Territory type check failed"
assert territory_dim.select("TerritoryID").distinct().count() == territory_dim.count(), "Duplicate territory keys"

if not spark.catalog.tableExists("adventureworks.silver.dim_sales_territory"):
    territory_dim.write.saveAsTable("adventureworks.silver.dim_sales_territory")

print("Silver territory rows:", spark.table("adventureworks.silver.dim_sales_territory").count())

## 4. Gold: Build the purchasing fact
Join Silver purchase-order detail to its header on PurchaseOrderID to add VendorID
and OrderDate.
- Grain: one row per purchase-order line, exactly as assigned.
- Check header and line-key uniqueness.
- Confirm the join preserves the detail row count.
- Check product and vendor references against their dimensions.
- Saved table: gold.fact_purchases, 8,845 rows.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS adventureworks.gold")

pod = spark.table("adventureworks.silver.stg_purchase_order_detail")
poh = spark.table("adventureworks.silver.stg_purchase_order_header").select("PurchaseOrderID", "VendorID", "OrderDate")
assert poh.select("PurchaseOrderID").distinct().count() == poh.count(), "Duplicate header keys"
assert pod.select("PurchaseOrderID", "PurchaseOrderDetailID").distinct().count() == pod.count(), "Duplicate purchase lines"

fact_purchases = pod.join(poh, "PurchaseOrderID", "inner")
assert fact_purchases.count() == pod.count(), "Join lost or multiplied purchase lines"

products = spark.table("adventureworks.silver.dim_product")
vendors = spark.table("adventureworks.silver.dim_vendor")
assert fact_purchases.join(products, "ProductID", "left_anti").count() == 0, "Product missing from dimension"
assert fact_purchases.join(vendors, fact_purchases["VendorID"] == vendors["BusinessEntityID"], "left_anti").count() == 0, "Vendor missing from dimension"

if not spark.catalog.tableExists("adventureworks.gold.fact_purchases"):
    fact_purchases.write.saveAsTable("adventureworks.gold.fact_purchases")

print("Gold purchasing rows:", spark.table("adventureworks.gold.fact_purchases").count())

### Build the sales fact
Join the existing sales lines to the typed sales header on SalesOrderID, adding
OrderDate and TerritoryID.
- Grain: one row per sales-order line.
- Keep sales LineTotal at decimal(38,6) to retain the discount-calculation
  precision of the source.
- Check header and line-key uniqueness, join row counts, and product/territory
  references.
- Saved table: gold.fact_sales, 121,317 rows.

In [0]:
sales_lines = spark.table("adventureworks.bronze.salesorderdetail").selectExpr(
    "cast(SalesOrderID as int) as SalesOrderID",
    "cast(SalesOrderDetailID as int) as SalesOrderDetailID",
    "cast(ProductID as int) as ProductID",
    "cast(OrderQty as smallint) as OrderQty",
    "cast(UnitPrice as decimal(19,4)) as UnitPrice",
    "cast(UnitPriceDiscount as decimal(19,4)) as UnitPriceDiscount",
    "cast(LineTotal as decimal(38,6)) as LineTotal"
)
sales_dates = spark.table("adventureworks.silver.stg_sales_order_header").select("SalesOrderID", "OrderDate", "TerritoryID")
assert sales_dates.select("SalesOrderID").distinct().count() == sales_dates.count(), "Duplicate sales header keys"
assert sales_lines.select("SalesOrderID", "SalesOrderDetailID").distinct().count() == sales_lines.count(), "Duplicate sales lines"
assert sales_lines.where("SalesOrderID IS NULL OR ProductID IS NULL OR OrderQty IS NULL OR LineTotal IS NULL").count() == 0, "Missing sales key or measure"

fact_sales = sales_lines.join(sales_dates, "SalesOrderID", "inner")
assert fact_sales.count() == sales_lines.count(), "Join lost or multiplied sales lines"
assert fact_sales.join(spark.table("adventureworks.silver.dim_product"), "ProductID", "left_anti").count() == 0, "Product missing from dimension"
assert fact_sales.join(spark.table("adventureworks.silver.dim_sales_territory"), "TerritoryID", "left_anti").count() == 0, "Territory missing from dimension"

if not spark.catalog.tableExists("adventureworks.gold.fact_sales"):
    fact_sales.write.saveAsTable("adventureworks.gold.fact_sales")

print("Gold sales rows:", spark.table("adventureworks.gold.fact_sales").count())

## 5. Query 1: Five products with the lowest margin per unit
Unit price per product = sum(LineTotal) / sum(OrderQty), quantity-weighted; sales
LineTotal includes line discounts. Margin per unit = net sales unit price minus
purchase unit price.
Each fact is aggregated per product before the join, so transaction rows are never
multiplied. Only the 76 products present in both facts are compared.
The five lowest-margin ProductIDs are 948, 907, 937, 932 and 940; all five margins
are negative under this metric. These are historical average-price comparisons,
not margins from matched purchase and sale transactions.

In [0]:
q1_sales = spark.table("adventureworks.gold.fact_sales")
q1_purchases = spark.table("adventureworks.gold.fact_purchases")
assert q1_sales.where("OrderQty IS NULL OR OrderQty <= 0 OR LineTotal IS NULL").count() == 0, "Invalid sales quantity or amount"
assert q1_purchases.where("OrderQty IS NULL OR OrderQty <= 0 OR LineTotal IS NULL").count() == 0, "Invalid purchase quantity or amount"

sales_prices = q1_sales.groupBy("ProductID").sum("LineTotal", "OrderQty").selectExpr(
    "ProductID",
    "`sum(LineTotal)` / `sum(OrderQty)` AS UnitNetSalesPrice"
)
purchase_prices = q1_purchases.groupBy("ProductID").sum("LineTotal", "OrderQty").selectExpr(
    "ProductID",
    "`sum(LineTotal)` / `sum(OrderQty)` AS UnitPurchasePrice"
)
margins = sales_prices.join(purchase_prices, "ProductID").selectExpr(
    "ProductID", "UnitNetSalesPrice", "UnitPurchasePrice",
    "UnitNetSalesPrice - UnitPurchasePrice AS MarginPerUnit"
)
print("Products present in both facts:", margins.count())
margins.orderBy("MarginPerUnit", "ProductID").show(5, truncate=False)


## 6. Query 2: Monthly sales versus purchase quantities
Aggregate order quantities for sales and purchases by ProductID and calendar
month. FULL JOIN the aggregates and fill the missing side with zero, keeping
months present on either side, then filter for UnitsSold greater than
UnitsPurchased.
- Result: 3,727 product-months satisfy the condition.
- Example: Product 707 in May 2022 shows 24 units sold and 0 units ordered for
  purchase in that month.
"Purchased" means purchase-order quantity, not confirmed stock receipts. The
comparison alone does not prove a stock shortage.

In [0]:
q2_sales = spark.table("adventureworks.gold.fact_sales")
q2_purchases = spark.table("adventureworks.gold.fact_purchases")

sold = q2_sales.selectExpr("ProductID", "cast(OrderQty as int) AS Qty", "date_trunc('month', OrderDate) AS Month")
sold = sold.groupBy("ProductID", "Month").sum("Qty").withColumnRenamed("sum(Qty)", "UnitsSold")

bought = q2_purchases.selectExpr("ProductID", "cast(OrderQty as int) AS Qty", "date_trunc('month', OrderDate) AS Month")
bought = bought.groupBy("ProductID", "Month").sum("Qty").withColumnRenamed("sum(Qty)", "UnitsPurchased")

monthly = sold.join(bought, ["ProductID", "Month"], "full").fillna(0, ["UnitsSold", "UnitsPurchased"])
monthly = monthly.withColumn("Month", monthly["Month"].cast("date"))
faster = monthly.where("UnitsSold > UnitsPurchased")
print("Product-months where sales outran purchases:", faster.count())
faster.orderBy("Month", "ProductID").show(20, truncate=False)
